In [ ]:
import pandas as pd
import joblib
import sys
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor  # Added for comparison

sys.path.append(os.path.abspath('../src'))
from preprocessing import preprocess_data
from evaluate import get_metrics

# load and preprocess
df = pd.read_csv('../data/processed_engineered.csv')
df_final = preprocess_data(df)

# setup features
# drop redundant terms to ensure the model follows physics laws, not just correlations
features_to_remove = ['Speed_kmh', 'Speed_ms', 'Kinetic_Energy_Proxy', 'Aero_Power_Demand']
X = df_final.drop(columns=['Energy_Consumption_kWh'] + features_to_remove)
y = df_final['Energy_Consumption_kWh']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# model comparison

# train physics-optimised linear regression (champion)
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)

# train XGBoost (baseline comparison)
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)

# evaluation

print("--- MODEL COMPARISON RESULTS ---")
lr_metrics = get_metrics(y_test, lr_preds, "Linear Regression (Physics-Optimized)")
xgb_metrics = get_metrics(y_test, xgb_preds, "XGBoost (Black-Box)")

# create a small summary table for the report
results_df = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R2'],
    'Linear Regression': [lr_metrics['MAE'], lr_metrics['RMSE'], lr_metrics['R2']],
    'XGBoost': [xgb_metrics['MAE'], xgb_metrics['RMSE'], xgb_metrics['R2']]
})
print("\nFinal Comparison Table:")
print(results_df)

# champion selection
# save the linear regression model even if XGBoost has a higher R2.
# reason: physics-consistency, interpretability, and stable extrapolation.

joblib.dump(lr_model, '../models/ev_model.pkl')
joblib.dump(X.columns.tolist(), '../models/model_columns.pkl')
print("\nChampion Model (Linear Regression) saved successfully!")

--- MODEL COMPARISON RESULTS ---
Linear Regression (Physics-Optimized) Performance:
  MAE: 0.6018845523909213
  RMSE: 0.737827504429653
  R²: 0.8877375127845032
  MAPE: 8.014297285942986%
XGBoost (Black-Box) Performance:
  MAE: 0.5735369073248717
  RMSE: 0.7107730501923761
  R²: 0.8958193928868806
  MAPE: 7.4531024564657375%

Final Comparison Table:
  Metric  Linear Regression   XGBoost
0    MAE           0.601885  0.573537
1   RMSE           0.737828  0.710773
2     R2           0.887738  0.895819

Champion Model (Linear Regression) saved successfully!
